In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

docs=[
    "papers are made using trees",
    "collection of those papers are made into books",
    "many people like to read books",
    "book give good knowledge",
    "reading books help improve your speaking skills"
]

query="how to improve my speaking skills"

doc_emb = model.encode(docs) 
query_emb = model.encode(query)

# Compute similarity
similarities = cosine_similarity(
    query_emb.reshape(1, -1),
    doc_emb
)[0]

# Print top results
for i, score in enumerate(similarities):
    print(f"{docs[i]}  (Score: {score:.3f})")

c:\Users\rahul\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2105.17it/s]


papers are made using trees  (Score: 0.084)
collection of those papers are made into books  (Score: 0.052)
many people like to read books  (Score: 0.079)
book give good knowledge  (Score: 0.228)
reading books help improve your speaking skills  (Score: 0.657)


## For the 3 PDFs

In [2]:
from pypdf import PdfReader

In [3]:

PDF_PATHS = [
    "test.pdf",
    "Natural-Language-Processing-Python (1).pdf",
    "learning-langchain-for-true-epub-9781098167288.pdf",
]

QUERY = "how to improve my speaking skills"

In [4]:
CHUNK_SIZE = 300

In [5]:
def extract_chunks(pdf_path: str, chunk_size: int) -> list[str]:
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        text = page.extract_text() or ""
        full_text += text + " "

    full_text = full_text.strip()
    chunks = [
        full_text[i : i + chunk_size]
        for i in range(0, len(full_text), chunk_size)
        if full_text[i : i + chunk_size].strip()
    ]
    return chunks

In [6]:
docs = []
sources = []   

for pdf_path in PDF_PATHS:
    chunks = extract_chunks(pdf_path, CHUNK_SIZE)
    for idx, chunk in enumerate(chunks):
        docs.append(chunk)
        sources.append(f"{pdf_path}  [chunk {idx + 1}]")

print(f"Total chunks across all PDFs: {len(docs)}\n")

Total chunks across all PDFs: 7967



In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_emb   = model.encode(docs)
query_emb = model.encode(QUERY)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2645.08it/s]


In [8]:
similarities = cosine_similarity(
    query_emb.reshape(1, -1),
    doc_emb
)[0]


TOP_N = 5

ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

print(f"Query: {QUERY}\n")
print(f"Top {TOP_N} results:\n" + "─" * 60)

for rank, (i, score) in enumerate(ranked[:TOP_N], 1):
    print(f"\n#{rank}  Score: {score:.3f}  |  {sources[i]}")
    print(docs[i][:200] + ("..." if len(docs[i]) > 200 else ""))

Query: how to improve my speaking skills

Top 5 results:
────────────────────────────────────────────────────────────

#1  Score: 0.463  |  test.pdf  [chunk 1894]
14314 (2023). 7  Ning Ding et al. “Enhancing chat language models by scaling high-quality instructional
conversations.” arXiv preprint arXiv:2305.14233 (2023).
8  Fred Jelinek et al. “Perplexity—a mea...

#2  Score: 0.417  |  test.pdf  [chunk 1272]
live in a vacuum. As an example, your
body language, facial expressions, intonation, etc. are all methods of
communication that enhance the spoken word.
The same thing applies to LLMs; if we can enabl...

#3  Score: 0.399  |  Natural-Language-Processing-Python (1).pdf  [chunk 416]
 1 Packets of thought (NLP overview)
    You’ll use both approaches
(neural networks and hand-coded
algorithms) as you incrementally
build an NLP pipeline for a chat-
bot capable of conversing within ...

#4  Score: 0.381  |  Natural-Language-Processing-Python (1).pdf  [chunk 3114]
engine. You’ll build an